In [4]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('online_retail_II.csv', encoding='ISO-8859-1')

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [13]:
# canceled ones
df = df[~df['Invoice'].astype(str).str.startswith('C')]
len(df)
# 0 quantity
df = df[df['Quantity'] > 0]

In [12]:
basket = (df
          .groupby(['Invoice', 'Description'])['Quantity']
          .sum()
          .unstack()
          .fillna(0))

# Convert to binary (0/1)
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

basket.head()

Description,DOORMAT UNION JACK GUNS AND ROSES,3 STRIPEY MICE FELTCRAFT,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,ANIMAL STICKERS,BLACK PIRATE TREASURE CHEST,BROWN PIRATE TREASURE CHEST,Bank Charges,CAMPHOR WOOD PORTOBELLO MUSHROOM,CHERRY BLOSSOM DECORATIVE FLASK,...,wrongly coded 20713,wrongly coded 23343,wrongly coded-23343,wrongly marked,wrongly marked 23343,wrongly marked carton 22804,wrongly marked. 23343 in box,wrongly sold (22719) barcode,wrongly sold as sets,wrongly sold sets
Invoice,,,,,,,,,,,,,,,,,,,,,
489434,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
489435,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
489436,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
489437,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
489438,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
import seaborn as sns
import matplotlib.pyplot as plt

top_items = df['Description'].value_counts().head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=top_items.values, y=top_items.index)
plt.title("Top 10 Most Purchased Items")
plt.xlabel("Frequency")
plt.ylabel("Items")
plt.show()

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
top_items_list = top_items.index
subset = basket[top_items_list]

corr = subset.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Item Co-occurrence Heatmap")
plt.show()

In [ ]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets_ap = apriori(
    basket,
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets_ap.head()

In [ ]:
top_ap = frequent_itemsets_ap.sort_values(by='support', ascending=False).head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=top_ap['support'], y=top_ap['itemsets'].astype(str))
plt.title("Top Frequent Itemsets (Apriori)")
plt.show()

In [ ]:
from mlxtend.frequent_patterns import fpgrowth

frequent_itemsets_fp = fpgrowth(
    basket,
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets_fp.head()

In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets_ap,
    metric="confidence",
    min_threshold=0.5
)

rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head()

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=rules,
    x='confidence',
    y='lift',
    size='support',
    hue='lift'
)
plt.title("Confidence vs Lift")
plt.show()

In [ ]:
import time

start = time.time()
apriori(basket, min_support=0.01, use_colnames=True)
ap_time = time.time() - start

start = time.time()
fpgrowth(basket, min_support=0.01, use_colnames=True)
fp_time = time.time() - start

print("Apriori Time:", ap_time)
print("FP-Growth Time:", fp_time)